# Đề tài 10 · Nhận dạng mẫu: Phân loại Âm thanh & Phát hiện Giọng nói Nhân bản Zero-Shot
## Thực nghiệm 4.0: Phân tích Sai số & Đánh giá Thính giác Ẩn danh (Error Analysis & Blind Perceptual Test)

**Nhóm nghiên cứu: Nhóm 05 · Chuyên ngành Khoa học Máy tính / Trí tuệ Nhân tạo**

### 1. Mục tiêu Nghiên cứu
Nghiên cứu sâu cơ chế sai số của bộ phân loại phát hiện giọng giả mạo T-A và tiến hành khảo sát thính giác nhận thức của con người đối với âm thanh nhân bản zero-shot:
- Phân rã sai số của mô hình SVM T-A theo người nói, theo hệ thống sinh âm thanh và theo từng dải đặc trưng âm học.
- Tiến hành thử nghiệm nghe mù ẩn danh (Blind Perceptual Test) trên tập mẫu kiểm thử để đo lường độ chân thực tự nhiên (Naturalness) và mức độ tương đồng chất giọng (Speaker Similarity) từ góc độ người nghe thực tế.

### 2. Thiết kế Giao thức Thực nghiệm (Experimental Protocol)
- **Mẫu thính giác ẩn danh:** 14 câu thoại phân bố đều trên 7 người nói thuộc tập Test độc lập (2 câu/người nói). Mỗi câu thoại gồm 1 bản ghi thật chuẩn và 3 bản ghi tổng hợp từ 3 mô hình được hoán vị ngẫu nhiên (A/B/C) để loại trừ định kiến chủ quan.
- **Tiêu chí đánh giá nhận thức:**
  1. *Độ chính xác ngữ âm (Phonetic Accuracy):* Ghi nhận các hiện tượng đọc sai dấu, nuốt âm, vấp từ hoặc biến dạng âm tiết.
  2. *Điểm tương đồng chất giọng (Similarity Score):* Đánh giá tương quan âm sắc giữa mẫu sinh và giọng nói tham chiếu gốc (thang điểm MOS 1–5).
  3. *Điểm tự nhiên (Naturalness Score):* Đánh giá nhịp điệu (prosody) và tính trôi chảy tự nhiên của ngữ điệu tiếng Việt.

### 3. Đóng góp Khoa học & Sản phẩm
- Báo cáo phân tích ma trận nhầm lẫn và phân bố điểm quyết định siêu phẳng của mô hình SVM.
- Bộ dữ liệu đánh giá thính giác nhận thức chuẩn hóa kèm giao diện tương tác HTML tự chứa (`NGHE_MU_TA.html` và `BAO_CAO_PHAN_TICH_TA.html`).


In [ ]:
# Ô 1 · NẠP OUTPUT CỦA BƯỚC 3 — CHỈ CPU
import os,sys,subprocess,importlib.util,importlib,json,random,hashlib,base64,io,html,zipfile,shutil,time
from pathlib import Path
for x in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):os.environ[x]='1'
def can(pkg,mod):
    if importlib.util.find_spec(mod) is not None:return
    print('Cài',pkg,'— cần Internet nếu chưa có sẵn')
    p=subprocess.run([sys.executable,'-m','pip','install','-q',pkg],capture_output=True,text=True)
    importlib.invalidate_caches()
    if p.returncode!=0 or importlib.util.find_spec(mod) is None:raise RuntimeError('Không cài được '+pkg+': '+(p.stderr or p.stdout)[-1200:])
for a,b in [('numpy','numpy'),('pandas','pandas'),('scikit-learn','sklearn'),('matplotlib','matplotlib'),('soundfile','soundfile')]:can(a,b)
import numpy as np,pandas as pd,soundfile as sf,sklearn,matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score,accuracy_score,f1_score,roc_auc_score,confusion_matrix
NGUON=Path('/kaggle/input/notebooks/ta-vivos-thatgia-tudong/ta_vivos')
if not (NGUON/'manifest_TA.csv').is_file():
    raise FileNotFoundError('Không thấy '+str(NGUON/'manifest_TA.csv')+'. Add Input output notebook ta-vivos-thatgia-tudong (cần thư mục ta_vivos, không chỉ ZIP).')
RA=Path('/kaggle/working/buoc4_TA');RA.mkdir(parents=True,exist_ok=True)
M=pd.read_csv(NGUON/'manifest_TA.csv');F=pd.read_csv(NGUON/'dac_trung_TA.csv');design=pd.read_csv(NGUON/'thiet_ke_cap.csv');PRED=pd.read_csv(NGUON/'du_doan_test_TA.csv');INFO=json.loads((NGUON/'thong_tin_TA.json').read_text(encoding='utf-8'))
print('✓ Input:',NGUON,'·',len(M),'clip ·',M.pair_id.nunique(),'cặp · Python',sys.version.split()[0])
if M.id_clip.duplicated().any() or F.id_clip.duplicated().any() or set(M.id_clip)!=set(F.id_clip):raise ValueError('Manifest và bảng đặc trưng không khớp một–một.')
if not set(M.tap).issubset({'train','test'}):raise ValueError('Tên tập không hợp lệ')
if set(M[M.tap=='train'].speaker)&set(M[M.tap=='test'].speaker):raise ValueError('Người train/test bị trùng')
if not M.groupby('pair_id').size().eq(4).all():raise ValueError('Có cặp không đủ bốn nguồn')
if not M.sr.eq(16000).all():raise ValueError('Tần số manifest không đồng nhất 16 kHz')
if not (M.groupby('pair_id').nguon.apply(set).apply(lambda s:s=={'that','vieneu','omnivoice','omdiom'})).all():raise ValueError('Cặp thiếu nguồn hoặc trùng nguồn')
D=M.merge(F,on='id_clip',validate='one_to_one');FC=sorted(c for c in F if c.startswith(('F1_','F2_','F3_')))
if len(FC)!=75 or not np.isfinite(D[FC].to_numpy(float)).all():raise ValueError('Thiếu 75 đặc trưng hữu hạn')
print('✓ 75 đặc trưng, speaker train/test tách biệt; OmDiOm đã fine-tune trên VIVOS theo xác nhận người dùng.')


In [ ]:
# Ô 2 · PHÂN TÍCH KẾT QUẢ CŨ — KHÔNG DÙNG TEST ĐỂ CHỌN LẠI MÔ HÌNH
tr=D[D.tap=='train'].copy();te=D[D.tap=='test'].copy();te_b=te[te.test_can_bang.astype(str).str.lower().isin(['true','1'])].copy()
if tr.speaker.nunique()<3 or te.speaker.nunique()<1:raise ValueError('Không đủ speaker để phân tích')
obs=PRED[PRED.tap_test=='toan_bo_1_that_3_gia'].copy()
if len(obs)!=len(te) or set(obs.id_clip)!=set(te.id_clip):raise ValueError('Dự đoán test và manifest không khớp')
obs['sai']=(obs.nhan!=obs.du_doan)
wrong=obs[obs.sai].merge(te[['id_clip','text','giay']],on='id_clip',validate='one_to_one')
wrong.to_csv(RA/'nhung_clip_du_doan_sai.csv',index=False,encoding='utf-8-sig')
speakers=[]
for sp,g in obs.groupby('speaker'):
    cm=confusion_matrix(g.nhan,g.du_doan,labels=[0,1]);specificity=cm[0,0]/cm[0].sum();sensitivity=cm[1,1]/cm[1].sum()
    speakers.append({'speaker':sp,'clip':len(g),'that_bi_nhan_gia':int(cm[0,1]),'gia_bi_nhan_that':int(cm[1,0]),'balanced_accuracy':float((specificity+sensitivity)/2),'macro_f1':float(f1_score(g.nhan,g.du_doan,average='macro',zero_division=0))})
S=pd.DataFrame(speakers);S.to_csv(RA/'ket_qua_tung_nguoi.csv',index=False,encoding='utf-8-sig')
fake=obs[obs.nhan==1];G=fake.assign(phat_hien=(fake.du_doan==1).astype(int)).groupby('nguon',as_index=False).agg(so_clip=('phat_hien','size'),dung=('phat_hien','sum'),recall_gia=('phat_hien','mean'))
G.to_csv(RA/'ket_qua_tung_bo_sinh.csv',index=False,encoding='utf-8-sig')
# Xấp xỉ bootstrap theo speaker của mô hình đã cố định, không bao gồm bất định chọn mô hình.
# Chỉ 7 speaker: khoảng này không phải bảo đảm cho cộng đồng người nói.
rng=np.random.default_rng(20261103);spks=sorted(obs.speaker.unique());cms=[]
for sp in spks:
    q=obs[obs.speaker==sp];cms.append(confusion_matrix(q.nhan,q.du_doan,labels=[0,1]))
cms=np.stack(cms);bs=[]
for _ in range(4000):
    cm=cms[rng.integers(0,len(spks),len(spks))].sum(axis=0)
    bs.append(.5*(cm[0,0]/cm[0].sum()+cm[1,1]/cm[1].sum()))
ci=np.quantile(bs,[.025,.975]); ci_data={'n_speaker':len(spks),'so_mau_lay_lai':4000,'seed':20261103,'loai':'bootstrap theo speaker test cho mo hinh da co dinh; khong bao gom chon mo hinh/huan luyen','balanced_accuracy_cua_mo_hinh_da_fit':float(balanced_accuracy_score(obs.nhan,obs.du_doan)),'can_duoi_2_5':float(ci[0]),'can_tren_97_5':float(ci[1])}
(RA/'bootstrap_theo_nguoi.json').write_text(json.dumps(ci_data,ensure_ascii=False,indent=2),encoding='utf8')
print('✓',len(wrong),'clip sai; bootstrap theo',len(spks),'người, mô hình đã cố định:',ci.round(4))


In [ ]:
# Ô 3 · THĂM DÒ NHÓM ĐẶC TRƯNG — CÙNG SVM, TRAIN THEO NGƯỜI
# Không chọn nhóm bằng test; nhóm F1 (MFCC) vẫn chứa thông tin phổ.
nhom={'F1+F2+F3':FC,'F1+F3':[c for c in FC if c.startswith(('F1_','F3_'))],
      'F1':[c for c in FC if c.startswith('F1_')], 'F2':[c for c in FC if c.startswith('F2_')], 'F3':[c for c in FC if c.startswith('F3_')]}
folds=list(GroupKFold(n_splits=3).split(tr,groups=tr.speaker));cv=[];ab=[]
for name,cols in nhom.items():
    for fold,(a,b) in enumerate(folds,1):
        tt=tr.iloc[a];vv=tr.iloc[b]
        if not set(tt.speaker).isdisjoint(set(vv.speaker)):raise ValueError('Rò rỉ người trong CV')
        model=make_pipeline(StandardScaler(),SVC(C=1,kernel='rbf',class_weight='balanced'))
        model.fit(tt[cols].to_numpy(float),tt.nhan.to_numpy());pr=model.predict(vv[cols].to_numpy(float))
        cv.append({'nhom':name,'so_dac_trung':len(cols),'fold':fold,'balanced_accuracy':balanced_accuracy_score(vv.nhan,pr),'macro_f1':f1_score(vv.nhan,pr,average='macro',zero_division=0)})
    model=make_pipeline(StandardScaler(),SVC(C=1,kernel='rbf',class_weight='balanced'))
    model.fit(tr[cols].to_numpy(float),tr.nhan.to_numpy())
    yy=te.nhan.to_numpy();pred=model.predict(te[cols].to_numpy(float));score=model.decision_function(te[cols].to_numpy(float));mat=confusion_matrix(yy,pred,labels=[0,1])
    ab.append({'nhom':name,'so_dac_trung':len(cols),'cv_balanced_accuracy_trung_binh':float(np.mean([z['balanced_accuracy'] for z in cv if z['nhom']==name])),
               'test_balanced_accuracy_tham_do':float(balanced_accuracy_score(yy,pred)),'test_macro_f1_tham_do':float(f1_score(yy,pred,average='macro',zero_division=0)),
               'test_auc_tham_do':float(roc_auc_score(yy,score)),'that_bi_nhan_gia':int(mat[0,1]),'gia_bi_nhan_that':int(mat[1,0])})
CV=pd.DataFrame(cv);AB=pd.DataFrame(ab);CV.to_csv(RA/'tham_do_dac_trung_CV_train.csv',index=False,encoding='utf-8-sig');AB.to_csv(RA/'tham_do_dac_trung_test.csv',index=False,encoding='utf-8-sig')
base=float(balanced_accuracy_score(obs.nhan,obs.du_doan));reproduced=AB.loc[AB.nhom=='F1+F2+F3','test_balanced_accuracy_tham_do'].iloc[0]
if abs(base-reproduced)>1e-9:print('⚠️ Tái huấn luyện all-features khác model đã lưu:',base,reproduced,'Kiểm tra phiên bản thư viện.')
print('✓ Bảng thăm dò nhóm đặc trưng (không lựa chọn dựa vào test):');print(AB.round(4).to_string(index=False))


In [ ]:
# Ô 4 · CHỌN 14 CẶP TEST MỚI ĐỂ NGHE, THAM CHIẾU NHÃN ĐƯỢC XÁO TRỘN
seed=20261104;rng=random.Random(seed)
# Ba mẫu đã có sẵn trong HTML Bước 3, tránh nghe lại nếu có đủ cặp.
da_nghe={'VIVOSSPK03__VIVOSSPK03_R038','VIVOSSPK19__VIVOSSPK19_012','VIVOSSPK20__VIVOSSPK20_054'}
meta=te[te.nguon=='that'].sort_values(['speaker','pair_id']).copy();chosen=[]
for sp,g in meta.groupby('speaker',sort=True):
    g=g[~g.pair_id.isin(da_nghe)].sort_values(['giay','pair_id']).reset_index(drop=True)
    if len(g)<2:raise ValueError('Không đủ hai câu chưa từng nhúng trong báo cáo cũ cho '+sp)
    # Chọn một từ nửa thời lượng ngắn, một từ nửa dài; deterministic, không chọn theo lỗi model.
    mid=len(g)//2;low=g.iloc[:mid];high=g.iloc[mid:]
    chosen.extend([low.iloc[rng.randrange(len(low))],high.iloc[rng.randrange(len(high))]])
if len(chosen)!=14:raise ValueError(f'Cần 7 người test × 2 câu; hiện có {len(chosen)} câu. Không ghi báo cáo 14 câu sai thực tế.')
C=pd.DataFrame(chosen);C.to_csv(RA/'mau_nghe_14_cap.csv',index=False,encoding='utf-8-sig')
# Kiểm tra file input mới thay vì đường dẫn /kaggle/working được lưu trong manifest cũ.
def audio_path(r):return NGUON/'audio'/r.nguon/r.speaker/(r.cau_id+'.flac')
def audio_data(r):
    p=audio_path(r)
    if not p.is_file():raise FileNotFoundError('Thiếu audio '+str(p)+'; Add Input toàn bộ output thư mục ta_vivos, không chỉ CSV.')
    inf=sf.info(p)
    if inf.samplerate!=16000 or inf.channels!=1 or inf.frames<=0:raise ValueError('Audio không hợp lệ: '+str(p))
    return 'data:audio/flac;base64,'+base64.b64encode(p.read_bytes()).decode('ascii')
items=[]
for i,row in enumerate(C.itertuples(index=False),1):
    z=te[te.pair_id==row.pair_id];real=z[z.nguon=='that'].iloc[0]
    sources=['vieneu','omnivoice','omdiom'];rng.shuffle(sources)
    fakes=[]
    for letter,b in zip(('A','B','C'),sources):
        z1=z[z.nguon==b]
        if len(z1)!=1:raise ValueError('Không đủ clip tại '+row.pair_id+' / '+b)
        fakes.append({'letter':letter,'bo':b,'audio':audio_data(z1.iloc[0])})
    items.append({'stt':i,'pair_id':row.pair_id,'speaker':row.speaker,'cau_id':row.cau_id,'text':row.text,'real_audio':audio_data(real),'fakes':fakes})
eval_id=hashlib.sha256(json.dumps([{'pair_id':q['pair_id'],'thu_tu':[x['bo'] for x in q['fakes']]} for q in items],sort_keys=True).encode()).hexdigest()[:16]
print('✓ Đã chọn',len(items),'cặp:',len(items)*4,'clip; mã phiếu:',eval_id,'· ba mẫu báo cáo cũ được tránh')


In [ ]:
# Ô 5 · HTML NGHE ẨN NHÃN TỰ CHỨA; XUẤT CSV SAU KHI ĐÁNH GIÁ
# Mã nguồn HTML có mapping nên không phải blind bảo mật. JSON lưu dở không ghi tên model.
style="""body{margin:0;background:#f1f6f8;color:#1a3540;font:16px/1.5 system-ui,Segoe UI,sans-serif}header{background:linear-gradient(120deg,#123d4e,#277b7d);color:#fff;padding:32px 24px}header div,main{max-width:1100px;margin:auto}main{padding:18px 20px 70px}h1{margin:0}h2{color:#176575}.notice{background:#fff6df;border-left:4px solid #df9f23;padding:15px}.card{margin:18px 0;background:white;border:1px solid #dae7eb;border-radius:14px;padding:18px}.row{display:grid;grid-template-columns:repeat(auto-fit,minmax(230px,1fr));gap:14px}.box{background:#f8fbfc;border:1px solid #d9e8ec;border-radius:10px;padding:12px}audio,select,textarea,input{width:100%;box-sizing:border-box;margin:6px 0;font:inherit}select,input,textarea{padding:8px;border:1px solid #b9cfd5;border-radius:6px;background:white}textarea{height:65px}button{background:#12697b;color:white;border:0;border-radius:7px;padding:10px 14px;cursor:pointer;margin:6px 6px 6px 0}button.secondary{background:#486675}button:disabled{opacity:.45;cursor:not-allowed}.small{font-size:14px;color:#4d6870}label{display:block;margin-top:8px}.status{position:sticky;top:0;z-index:2;padding:8px 14px;background:#daf0ed;border:1px solid #b4dfd8}#reveal{font-weight:bold;color:#7b3922}.controls{display:flex;flex-wrap:wrap;gap:4px}.file{max-width:400px}table{border-collapse:collapse;width:100%}td,th{border-bottom:1px solid #ddd;padding:6px;text-align:left}"""
h=[];h.append("<!doctype html><html lang='vi'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'><title>Nghe so sánh giọng T-A</title><style>"+style+"</style></head><body><header><div><h1>Nghe so sánh ẩn tên bộ sinh</h1><p>14 câu · 7 người · bản thật và ba bản A / B / C</p></div></header><main>")
h.append("<div class='notice'><b>Cách làm:</b> Mở tệp HTML đã tải xuống bằng Chrome hoặc Edge; nghe bản thật rồi từng bản A/B/C. Trả lời cho từng bản: đọc đủ/sai chữ, tự nhiên và giống giọng. A/B/C được xáo trộn <b>độc lập theo từng câu</b> — A ở câu này không chắc là A ở câu khác. Bạn đã nghe một số mẫu trước đó nên đây không phải thí nghiệm mù nghiêm ngặt.</div>")
h.append("<div class='card'><label>Mã người nghe (tùy chọn, không cần tên thật)<input id='listener' placeholder='Ví dụ: N01'></label><div id='status' class='status'></div><div class='controls'><button id='save'>Lưu tiến độ JSON</button><button id='importBtn' class='secondary'>Mở tiến độ JSON</button><input type='file' id='importFile' class='file' accept='.json,application/json' style='display:none'><button id='export' disabled>Hoàn tất & tải kết quả CSV</button><button id='show' class='secondary' disabled>Hiện tên bộ sinh</button></div><p class='small'>Cần chọn <b>cả ba mục</b> cho đủ 42 bản giả trước khi xuất CSV hoặc hiện tên. Nếu chưa chắc, chọn “không rõ”. Lưu JSON khi muốn nghỉ; tránh chỉ dựa vào bộ nhớ trình duyệt. CSV làm lộ tên bộ sinh.</p><div id='reveal'></div></div>")
esc=html.escape
options_content=[('','— Chọn —'),('du_dung','Đủ chữ, đúng chữ'),('thieu','Thiếu chữ'),('sai','Sai chữ'),('thieu_va_sai','Vừa thiếu vừa sai'),('khong_ro','Không rõ')]
options_score=[('','— Chọn —'),('1','1 — rất kém'),('2','2 — kém'),('3','3 — trung bình'),('4','4 — tốt'),('5','5 — rất tốt'),('khong_ro','Không rõ')]
def opt(xs):return ''.join('<option value="'+esc(a,quote=True)+'">'+esc(b)+'</option>' for a,b in xs)
for item in items:
    h.append("<section class='card'><h2>Câu "+str(item['stt'])+" · "+esc(item['speaker'])+"</h2><p><b>Văn bản:</b> "+esc(item['text'])+"</p><div class='box'><b>Bản thật — nghe trước để đối chiếu</b><audio controls preload='none' src='"+item['real_audio']+"'></audio></div><div class='row'>")
    for q in item['fakes']:
        key=item['pair_id']+'|'+q['letter'];keyq=esc(key,quote=True)
        h.append("<div class='box'><b>Bản "+q['letter']+"</b><audio controls preload='none' src='"+q['audio']+"'></audio>"+
                 "<label>Nội dung so với văn bản<select data-key='"+keyq+"' data-field='noi_dung'>"+opt(options_content)+"</select></label>"+
                 "<label>Độ tự nhiên<select data-key='"+keyq+"' data-field='tu_nhien'>"+opt(options_score)+"</select></label>"+
                 "<label>Độ giống giọng bản thật<select data-key='"+keyq+"' data-field='giong'>"+opt(options_score)+"</select></label>"+
                 "<label>Ghi chú chữ bị thiếu/sai (tùy chọn)<textarea data-key='"+keyq+"' data-field='ghi_chu' placeholder='Ví dụ: thiếu cụm ...'></textarea></label></div>")
    h.append('</div></section>')
# Audio không nằm trong mapping để JS nhẹ; nguồn được dùng khi xuất CSV/giải mã.
map_items=[{'stt':q['stt'],'pair_id':q['pair_id'],'speaker':q['speaker'],'cau_id':q['cau_id'],'text':q['text'],'map':{f['letter']:f['bo'] for f in q['fakes']}} for q in items]
meta_js=json.dumps(map_items,ensure_ascii=False).replace('<','\\u003c');id_js=json.dumps(eval_id)
script=r"""
<script>
const ITEMS=__ITEMS__, EVAL_ID=__ID__, STORAGE='TA_NGHE_'+EVAL_ID;
let state={eval_id:EVAL_ID,listener:'',answers:{},revealed:false};
try {const p=JSON.parse(localStorage.getItem(STORAGE)||'null'); if(p&&p.eval_id===EVAL_ID)state=p;} catch(e){}
const get=(k)=>state.answers[k]||{};
function update(){
 const fields=document.querySelectorAll('[data-key]');
 fields.forEach(e=>{const v=get(e.dataset.key)[e.dataset.field];if(v!==undefined)e.value=v});
 document.getElementById('listener').value=state.listener||'';
 const keys=ITEMS.flatMap(x=>['A','B','C'].map(k=>x.pair_id+'|'+k));
 const n=keys.filter(k=>['noi_dung','tu_nhien','giong'].every(f=>Boolean(get(k)[f]))).length;
 document.getElementById('status').textContent='Đã hoàn thành '+n+' / '+keys.length+' bản giả · Mã phiếu '+EVAL_ID;
 document.getElementById('export').disabled=n!==keys.length;
 document.getElementById('show').disabled=n!==keys.length;
 if(state.revealed) displayReveal();
}
function persist(){try{localStorage.setItem(STORAGE,JSON.stringify(state));}catch(e){}}
document.querySelectorAll('[data-key]').forEach(e=>e.addEventListener('change',()=>{const k=e.dataset.key;state.answers[k]=state.answers[k]||{};state.answers[k][e.dataset.field]=e.value;persist();update()}));
document.getElementById('listener').addEventListener('input',e=>{state.listener=e.target.value;persist()});
function download(text,name,mime){const url=URL.createObjectURL(new Blob([text],{type:mime}));const a=document.createElement('a');a.href=url;a.download=name;document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),10000)}
document.getElementById('save').onclick=()=>download(JSON.stringify(state,null,2),'tien_do_nghe_'+EVAL_ID+'.json','application/json;charset=utf-8');
document.getElementById('importBtn').onclick=()=>document.getElementById('importFile').click();
document.getElementById('importFile').onchange=e=>{const f=e.target.files[0];if(!f)return;const r=new FileReader();r.onload=()=>{try{const a=JSON.parse(r.result);if(a.eval_id!==EVAL_ID||!a.answers||typeof a.answers!=='object')throw Error('Mã phiếu không khớp');state=a;persist();update();alert('Đã mở tiến độ.')}catch(err){alert('Không mở được tiến độ: '+err.message)}};r.readAsText(f,'UTF-8');e.target.value='';};
function csv(s){let t=String(s??'');if(/^[\s]*[=+@-]/.test(t))t="'"+t;return '"'+t.replace(/"/g,'""')+'"';}
document.getElementById('export').onclick=()=>{
 const rows=[['eval_id','nguoi_nghe','so_cau','pair_id','speaker','cau_id','text','nhan_an','bo_sinh','noi_dung','tu_nhien','giong','ghi_chu']];
 for(const q of ITEMS) for(const k of ['A','B','C']){const a=get(q.pair_id+'|'+k);rows.push([EVAL_ID,state.listener,q.stt,q.pair_id,q.speaker,q.cau_id,q.text,k,q.map[k],a.noi_dung,a.tu_nhien,a.giong,a.ghi_chu||'']);}
 download('\uFEFF'+rows.map(r=>r.map(csv).join(',')).join('\r\n'),'ket_qua_nghe_TA_'+EVAL_ID+'.csv','text/csv;charset=utf-8');
 state.revealed=true;persist();displayReveal();
};
function displayReveal(){const el=document.getElementById('reveal');el.textContent='Đã mở nhãn: '+ITEMS.map(q=>'Câu '+q.stt+': '+['A','B','C'].map(k=>k+' = '+q.map[k]).join(', ')).join(' | ');}
document.getElementById('show').onclick=()=>{if(confirm('Hiện tên sẽ làm mất việc ẩn nhãn. Bạn đã hoàn thành phiếu?')){state.revealed=true;persist();displayReveal()}};
update();
</script>"""
script=script.replace('__ITEMS__',meta_js).replace('__ID__',id_js)
h.append(script+'</main></body></html>');listen=RA/'NGHE_MU_TA.html';listen.write_text(''.join(h),encoding='utf-8')
print(f'✓ {listen} · {len(items)*4} audio nhúng · 42 bản giả cần chấm')


In [ ]:
# Ô 6 · BIỂU ĐỒ VÀ BÁO CÁO PHÂN TÍCH TỰ CHỨA
plt.rcParams.update({'figure.facecolor':'white','axes.spines.top':False,'axes.spines.right':False,'font.size':10.5,'axes.titleweight':'bold','figure.autolayout':True})
figs=[];(RA/'hinh').mkdir(exist_ok=True)
def plot(title,note,fig,name):
    b=io.BytesIO();fig.savefig(b,format='png',dpi=145,bbox_inches='tight',facecolor='white');plt.close(fig);data=b.getvalue();(RA/'hinh'/(name+'.png')).write_bytes(data);figs.append((title,note,base64.b64encode(data).decode('ascii')))
cm=confusion_matrix(obs.nhan,obs.du_doan,labels=[0,1]);norm=cm/cm.sum(axis=1,keepdims=True)
fig,ax=plt.subplots(figsize=(6,4));im=ax.imshow(norm,cmap='Blues',vmin=0,vmax=1)
for i in range(2):
    for j in range(2):ax.text(j,i,f'{cm[i,j]}\n{norm[i,j]*100:.1f}%',ha='center',va='center',fontsize=13,color='white' if norm[i,j]>.55 else '#203743')
ax.set_xticks([0,1],['thật','giả']);ax.set_yticks([0,1],['thật','giả']);ax.set_xlabel('Dự đoán');ax.set_ylabel('Nhãn thật');ax.set_title('Test toàn bộ · 105 thật / 315 giả');fig.colorbar(im,ax=ax,shrink=.75)
plot('Ma trận nhầm lẫn','Không thay đổi kết quả Bước 3. Mỗi hàng chuẩn hóa theo nhãn thật.',fig,'01_nham_lan')
fig,ax=plt.subplots(figsize=(8,4));ax.bar(S.speaker,S.balanced_accuracy,color='#207e92');ax.set_ylim(0,1.06);ax.tick_params(axis='x',rotation=40);ax.set_ylabel('Balanced accuracy');ax.set_title('Sai khác theo người test');ax.grid(axis='y',alpha=.2)
plot('Theo người nói','Mỗi người có 15 thật và 45 giả; chỉ 7 người. Không xem các clip cùng người là độc lập hoàn toàn.',fig,'02_speaker')
fig,ax=plt.subplots(figsize=(7,3.5));ax.bar(G.nguon,G.recall_gia,color=['#3ba089','#e4a05d','#6681b7']);ax.set_ylim(0,1.09);ax.set_ylabel('Tỉ lệ phát hiện lớp giả');ax.set_title('Recall giả theo từng bộ sinh')
for i,r in enumerate(G.itertuples(index=False)):ax.text(i,r.recall_gia+.015,f'{r.dung}/{r.so_clip} ({r.recall_gia:.1%})',ha='center')
plot('Theo bộ sinh','Recall giả không đo độ tự nhiên, đọc đủ chữ hay độ giống giọng; số đo chỉ phản ánh mẫu và thiết kế này.',fig,'03_bosinh')
fig,ax=plt.subplots(figsize=(9,4));n=np.arange(len(AB));ax.bar(n-.19,AB.cv_balanced_accuracy_trung_binh,width=.38,label='CV train, 3 fold theo người',color='#458a9b');ax.bar(n+.19,AB.test_balanced_accuracy_tham_do,width=.38,label='Test cũ — thăm dò',color='#e8a35c');ax.set_xticks(n,AB.nhom);ax.set_ylim(0,1.08);ax.set_ylabel('Balanced accuracy');ax.legend(frameon=False);ax.set_title('SVM cố định · các nhóm đặc trưng');ax.grid(axis='y',alpha=.2)
plot('Thăm dò nhóm đặc trưng','Không chọn cấu hình bằng test đã xem. F1 MFCC vẫn mang thông tin phổ; bỏ F2 không tương đương loại mọi dấu vết băng thông.',fig,'04_dactrung')
fig,ax=plt.subplots(figsize=(8,4));x=te.groupby('nguon')['F2_band_4000_8001_mean'].median().reindex(['that','vieneu','omnivoice','omdiom']);ax.bar(x.index,x.values,color=['#655a8d','#369a9d','#eb9e61','#58a77d']);ax.set_ylabel('Trung vị tỉ lệ năng lượng');ax.set_title('Tỉ lệ năng lượng 4–8 kHz · test');ax.grid(axis='y',alpha=.2)
plot('Thăm dò dải cao','Khác biệt là bằng chứng về phân bố đặc trưng, không chứng minh nguyên nhân hay đóng góp nhân quả; 16 kHz không loại mọi dấu vết nguồn.',fig,'05_pho')
# Báo cáo: không nhập nhầm đánh giá nghe chưa thực hiện.
def table(df):return df.to_html(index=False,classes='table',border=0,float_format=lambda x:f'{x:.4f}',escape=True)
style="""body{margin:0;background:#f2f6f8;color:#18323c;font:16px/1.55 system-ui,Segoe UI,sans-serif}header{background:linear-gradient(125deg,#123d4e,#207d80);color:white;padding:36px 20px}header div,main{max-width:1100px;margin:auto}main{padding:20px 20px 60px}h1{margin:0}h2{margin-top:36px;color:#176575}figure,.card{background:white;border:1px solid #dce7eb;border-radius:12px;padding:18px;overflow:auto;margin:16px 0}figure img{max-width:100%;display:block;margin:auto}figcaption{border-left:3px solid #3599a3;padding-left:12px;color:#526a72;font-size:14px}.warn{background:#fff5df;border-left:4px solid #e2a935;padding:16px}.table{border-collapse:collapse;white-space:nowrap;font-size:13px}.table td,.table th{padding:7px;border-bottom:1px solid #dfe9eb}.table th{background:#e7f3f4}"""
h=['<!doctype html><html lang="vi"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Phân tích T-A</title><style>',style,'</style></head><body><header><div><h1>Phân tích T-A · dữ liệu VIVOS</h1><p>Phân tích theo người, bộ sinh và nhóm đặc trưng · không sinh lại audio</p></div></header><main>']
h.append('<div class="warn"><b>Lưu ý cốt lõi:</b> Theo xác nhận của người dùng, OmDiOm được fine-tune trên VIVOS. Split theo người chỉ áp dụng cho bộ phân loại, không bảo đảm người hoặc câu test chưa có trong fine-tune OmDiOm. Test đã được xem, bảng thay đổi đặc trưng là thăm dò, không phải kiểm chứng độc lập.</div>')
h.append('<h2>Phạm vi dữ liệu</h2><div class="card"><p>'+str(M.pair_id.nunique())+' cặp đủ; '+str(len(M))+' clip; train '+str(tr.speaker.nunique())+' người, test '+str(te.speaker.nunique())+' người. Chỉ các cặp đủ và qua lọc sơ bộ được phân tích. Chất lượng nội dung chưa được chứng minh bởi ngưỡng âm lượng/thời lượng.</p><p>Người dùng đã phản ánh các clip OmniVoice đã nghe có hiện tượng thiếu hoặc sai chữ. Phản ánh này chưa phải tỷ lệ cho toàn bộ clip. Đánh giá nghe mới thực hiện trong tệp NGHE_MU_TA.html.</p></div>')
h.append('<h2>Sai số theo người và bộ sinh</h2><div class="card">'+table(S)+'</div><div class="card">'+table(G)+'</div>')
h.append('<h2>Thăm dò theo nhóm đặc trưng</h2><div class="card">'+table(AB)+'<p class="small">CV theo người trên train; test cũ chỉ là thăm dò sau khi đã xem. Không dùng các con số này để chọn kết quả tốt nhất rồi gọi đó là test độc lập.</p></div>')
h.append('<h2>Độ bất định có điều kiện</h2><div class="card"><p>Bootstrap theo 7 người test, mô hình đã fit cố định: balanced accuracy '+f"{ci_data['balanced_accuracy_cua_mo_hinh_da_fit']:.2%}"+'; khoảng bách phân vị 95% '+f"{ci[0]:.2%}–{ci[1]:.2%}"+'. Khoảng này không bao gồm bất định khi chọn/huấn luyện mô hình và có thể kém ổn định vì chỉ có 7 người; không bảo đảm phạm vi cho mọi người.</p></div>')
h.append('<h2>Biểu đồ</h2>')
for title,note,img in figs:h.append('<figure><h3>'+html.escape(title)+'</h3><img src="data:image/png;base64,'+img+'"><figcaption>'+html.escape(note)+'</figcaption></figure>')
h.append('<h2>Cách trình bày kết quả nghe</h2><div class="card"><p>Không có điểm nghe trong báo cáo này cho đến khi người nghe chấm và gửi CSV. Mẫu 14 câu chọn có chủ đích theo người và thời lượng, không phải mẫu ngẫu nhiên đại diện; người nghe đã biết nghiên cứu và đã nghe một số mẫu, do đó không xem đây là thử mù nghiêm ngặt. Cần báo rõ số người nghe và số mẫu.</p></div></main></body></html>')
report=RA/'BAO_CAO_PHAN_TICH_TA.html';report.write_text(''.join(h),encoding='utf8');print('✓ Báo cáo:',report,'·',len(figs),'hình')


In [ ]:
# Ô 7 · ĐÓNG GÓI; AUDIO ĐÃ NHÚNG TRONG HTML NGHE, KHÔNG SAO CHÉP 1.480 TỆP
readme="""# Hướng dẫn Bước 4 – phân tích và nghe

1. Tải `NGHE_MU_TA.html` về máy, mở bằng Chrome/Edge. Nghe bản thật rồi A/B/C cho từng câu. A/B/C được xáo trộn độc lập theo từng câu.
2. Trả lời đủ ba mục cho 42 bản giả. Nếu không chắc chọn Không rõ. Nhấn Lưu tiến độ JSON khi muốn nghỉ; có thể mở lại JSON đó trong chính HTML. Nhấn Hoàn tất & tải kết quả CSV sau khi chấm xong; khi ấy tên ba bộ sinh mới hiện ra.
3. Gửi file CSV kết quả nghe để tổng hợp. Không cần gửi lại ZIP audio 114 MiB.
4. `BAO_CAO_PHAN_TICH_TA.html` là báo cáo thăm dò bằng dữ liệu có sẵn; không tự đưa đánh giá nghe vào khi bạn chưa chấm.

Nghe ẩn nhãn không hoàn toàn mù: chuyên gia đã nghe thẩm định vài clip trước, có thể nhận ra giọng, và có thể xem mapping trong source HTML. Kết quả của một người nghe trên 14 cặp không chứng minh chất lượng tổng quát. Chọn mẫu theo người và hai vùng thời lượng, không phải mẫu ngẫu nhiên đại diện.

OmDiOm đã fine-tune trên VIVOS theo xác nhận người dùng; chưa đối chiếu danh sách train TTS với từng câu và người test. CV và test của bộ phân loại tách theo speaker nhưng không đồng nghĩa TTS chưa gặp VIVOS. Các nhóm đặc trưng dùng cùng test đã xem: phân tích thăm dò, không làm thay thế test độc lập.
"""
(RA/'HUONG_DAN_BUOC4.md').write_text(readme,encoding='utf-8')
(RA/'thong_tin_buoc4.json').write_text(json.dumps({'eval_id':eval_id,'nguon':str(NGUON),'seed':seed,'so_cap_nghe':len(items),'so_audio_nghe':len(items)*4,'speaker_test':int(te.speaker.nunique()),'omdiom_ft_vivos':'nguoi dung xac nhan','test_dac_trung':'tham do tren test da xem','sklearn':sklearn.__version__},ensure_ascii=False,indent=2),encoding='utf-8')
name='/kaggle/working/KET_QUA_BUOC4_TA_'+time.strftime('%m%d_%H%M%S');z=shutil.make_archive(name,'zip',RA)
print('\n📄 Phiếu nghe:',listen,'\n📊 Báo cáo:',report,'\n📦 ZIP:',z, f'({Path(z).stat().st_size/1024**2:.1f} MiB)')
print('Sau khi nghe: gửi ket_qua_nghe_TA_<ma>.csv. Không cần gửi lại dữ liệu âm thanh gốc.')
